# Quién se conecta con quién: una red de personajes, lugares y proyectos

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Objetivo

A partir del corpus real de Casciari, construir una red de qué personajes, lugares y proyectos aparecen juntos en el mismo párrafo, y usar esa red para ver quién ocupa el centro del relato. No es un recorrido por las herramientas de spaCy: es una pregunta de minería de texto, y cada herramienta entra en escena cuando la pregunta la necesita.

## Al terminar este cuaderno vas a poder:

1. Explicar qué es la coocurrencia y por qué sirve para construir una red de entidades a partir de texto.
2. Reutilizar una herramienta de spaCy ya construida y guardada (el pipeline del cuaderno `05`) para extraer entidades de un corpus real.
3. Contar quién aparece más, y con quién aparece más seguido.
4. Graficar esa red con `matplotlib`, sin depender de una librería de grafos.

> **Cómo usar este cuaderno.** No reemplaza a los ocho anteriores: los asume, y en particular asume el `05` — ahí se construyó la herramienta que acá solo se reutiliza. Esta vez no hay una parada por cada capacidad de spaCy: cada técnica aparece cuando la pregunta de la red la necesita, no antes. Entre paréntesis, en cada sección, está el cuaderno con el desarrollo completo de esa técnica.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Coocurrencia** | Que dos entidades aparezcan en el mismo tramo de texto — acá, el mismo párrafo. |
| **Red (o grafo)** | Una manera de representar conexiones: nodos unidos por aristas. |
| **Nodo** | Cada entidad de la red: un personaje, un lugar o un proyecto. |
| **Arista** | La conexión entre dos nodos. Acá, su peso es cuántas veces coincidieron. |
| **Pipeline personalizado** | El pipeline de spaCy con las reglas propias del cuaderno `05`, guardado en `pipeline_propio/`. |

## ⚙ Configuración del entorno

In [ ]:
# spacy: la biblioteca central del cuaderno.
import spacy

# componentes: registra "corrector_voseo" y "detector_personajes" en spaCy.
# Hace falta importarlo ANTES de cargar pipeline_propio (ver 05, sección 6).
import componentes

# pandas: para cargar el corpus y tabular resultados.
import pandas as pd

# numpy: para calcular las posiciones de los nodos en el gráfico de red.
import numpy as np

# Counter: para contar frecuencias e coocurrencias sin hacerlo a mano.
from collections import Counter

# matplotlib: para los dos gráficos del cuaderno.
import matplotlib.pyplot as plt

print("Librerías cargadas correctamente.")

## 1. La pregunta y los datos

Trabajamos con `datos/corpus.jsonl`: 5.428 párrafos reales, reconstruidos a partir de entradas del blog de Hernán Casciari publicadas entre 2004 y 2015 (ver `datos/FUENTE.md`). Una advertencia de esa misma fuente es central para este cuaderno: **la unidad de análisis es el párrafo, no el texto** — los límites entre los 386 textos originales se perdieron en la captura. Cuando digamos que dos entidades "coocurren", nos referimos a que aparecen en el mismo párrafo — no necesariamente en la misma escena del relato.

In [ ]:
corpus = pd.read_json("datos/corpus.jsonl", lines=True)

cantidad_de_parrafos = len(corpus)
print(f"Párrafos en el corpus: {cantidad_de_parrafos}")
corpus.head(3)

## 2. Una herramienta que ya construimos

No partimos del modelo de fábrica. En el cuaderno `05` se armó un pipeline propio: un `EntityRuler` cargado con `datos/personajes.txt` — una lista curada de personajes, lugares y proyectos del corpus, categorizados a mano — más un componente que corrige el voseo. Ese pipeline quedó guardado en `pipeline_propio/`. Lo reutilizamos tal cual.

Vale la pena recordar algo que ya viste en `05`: esa lista **es una decisión de análisis, no un archivo neutral**. Por ejemplo, "Mercedes" está cargada como `LUGAR` porque es la ciudad natal del autor, aunque también es un nombre de persona — así que en lo que sigue, toda mención de "Mercedes" va a contar como lugar, sin excepción.

In [ ]:
nlp = spacy.load("pipeline_propio")

print("pipeline cargado:", nlp.pipe_names)

## 3. Extraer entidades de muchos párrafos

Acá sí hace falta una palabra sobre escala (el desarrollo completo está en `06`): una red necesita muchos párrafos, no uno. Por eso usamos `nlp.pipe()` en vez de llamar a `nlp()` párrafo por párrafo — procesa por lotes y es notablemente más rápido.

Trabajamos sobre los primeros 1.500 párrafos: es el mismo recorte que se usó para construir `datos/personajes.txt`, así que la lista de entidades y el texto que vamos a recorrer están alineados. **Esta celda tarda uno 15 a 20 segundos** — son 1.500 párrafos pasando por un pipeline completo, no una oración suelta.

In [ ]:
subset_del_corpus = corpus.head(1500)
textos = list(subset_del_corpus["texto"])

entidades_por_parrafo = []
categoria_por_entidad = {}

for doc in nlp.pipe(textos, batch_size=50):
    entidades_del_parrafo = []
    for entidad in doc.ents:
        si_es_de_interes = entidad.label_ in ("PERSONAJE", "LUGAR", "PROYECTO")
        if si_es_de_interes:
            entidades_del_parrafo.append(entidad.text)
            categoria_por_entidad[entidad.text] = entidad.label_
    entidades_por_parrafo.append(entidades_del_parrafo)

print("Párrafos procesados:", len(entidades_por_parrafo))

In [ ]:
parrafos_con_entidades = 0
for lista_de_entidades in entidades_por_parrafo:
    if len(lista_de_entidades) > 0:
        parrafos_con_entidades += 1

cantidad_de_parrafos_procesados = len(entidades_por_parrafo)

print(f"Párrafos con al menos una entidad de interés: {parrafos_con_entidades}")
print(f"Sobre un total de: {cantidad_de_parrafos_procesados}")

De 1.500 párrafos, poco más de 1 de cada 4 menciona a alguno de los personajes, lugares o proyectos de nuestra lista. El resto es narración sin ninguna de esas entidades — y eso también dice algo: la lista es un recorte, no todo el corpus gira en torno a estos nombres.

## 4. Quién aparece más

In [ ]:
todas_las_menciones = []
for lista_de_entidades in entidades_por_parrafo:
    for entidad in lista_de_entidades:
        todas_las_menciones.append(entidad)

frecuencia_individual = Counter(todas_las_menciones)
catorce_mas_frecuentes = frecuencia_individual.most_common(14)

for entidad, veces in catorce_mas_frecuentes:
    categoria = categoria_por_entidad[entidad]
    print(f"{entidad:15s} {categoria:10s} {veces}")

In [ ]:
colores_por_categoria = {
    "PERSONAJE": "steelblue",
    "LUGAR": "darkorange",
    "PROYECTO": "seagreen",
}

nombres = []
conteos = []
colores_de_las_barras = []
for entidad, veces in catorce_mas_frecuentes:
    nombres.append(entidad)
    conteos.append(veces)
    categoria = categoria_por_entidad[entidad]
    colores_de_las_barras.append(colores_por_categoria[categoria])

figura, eje = plt.subplots(figsize=(8, 5))
eje.barh(nombres, conteos, color=colores_de_las_barras)
eje.set_xlabel("menciones")
eje.set_title("Personajes, lugares y proyectos más mencionados")
eje.invert_yaxis()
plt.tight_layout()
plt.show()

Azul es personaje, naranja es lugar, verde es proyecto. "Chiri" — el alter ego de Casciari en el blog — encabeza la lista, seguido de "España" y "Mercedes": ya desde la frecuencia se nota que este es un relato de desplazamiento, entre el pueblo de origen y el país de destino.

## 5. Quién aparece junto a quién

Una mención individual dice quién está presente. No dice con quién comparte escena. Para eso necesitamos, en cada párrafo, todos los pares posibles entre las entidades que aparecieron ahí — sin contar un par dos veces, y sin emparejar una entidad consigo misma.

In [ ]:
coocurrencias = Counter()

for lista_de_entidades in entidades_por_parrafo:
    entidades_unicas = sorted(set(lista_de_entidades))
    for entidad_a in entidades_unicas:
        for entidad_b in entidades_unicas:
            si_es_un_par_nuevo = entidad_a < entidad_b
            if si_es_un_par_nuevo:
                coocurrencias[(entidad_a, entidad_b)] += 1

print("Pares distintos encontrados:", len(coocurrencias))

"`entidad_a < entidad_b`" no compara importancia: es una comparación alfabética, y solo sirve para no contar el par ("Alex", "Lucas") una vez como tal y otra vez como ("Lucas", "Alex").

In [ ]:
quince_pares_mas_frecuentes = coocurrencias.most_common(15)

filas_de_pares = []
for par, veces_juntos in quince_pares_mas_frecuentes:
    entidad_a, entidad_b = par
    fila = {"entidad A": entidad_a, "entidad B": entidad_b, "veces juntas": veces_juntos}
    filas_de_pares.append(fila)

pd.DataFrame(filas_de_pares)

### ✎ Para pensar

"Alex" y "Lucas" son el par que más veces coincide. "Carla" y "Fernando" también aparecen casi siempre juntos. Ninguno de los dos pares está en la lista de los más mencionados individualmente de la sección anterior — se necesitaba mirar la coocurrencia para encontrarlos.

¿Qué hipótesis te arriesgás sobre quiénes son, antes de ir a buscar el texto original? Y una vez que armes una hipótesis: ¿cómo la comprobarías con lo que ya sabés hacer con spaCy — buscando los párrafos donde aparecen, por ejemplo?

## 6. Graficar la red

No usamos ninguna librería de grafos: con lo que ya tenemos — `numpy` para la geometría y `matplotlib` para dibujar — alcanza. La idea es simple: ubicamos cada nodo en un punto de un círculo, y trazamos una línea entre dos nodos por cada vez que coocurrieron, más gruesa cuantas más veces.

In [ ]:
nodos = []
for entidad, veces in catorce_mas_frecuentes:
    nodos.append(entidad)

posiciones_de_nodos = {}
cantidad_de_nodos = len(nodos)
posicion_actual = 0
for nodo in nodos:
    angulo = 2 * np.pi * posicion_actual / cantidad_de_nodos
    posiciones_de_nodos[nodo] = (np.cos(angulo), np.sin(angulo))
    posicion_actual += 1

print("nodos ubicados:", len(posiciones_de_nodos))

In [ ]:
figura, eje = plt.subplots(figsize=(8, 8))

# Las aristas: una línea por par, solo entre nodos que están en el gráfico.
for par, veces_juntos in coocurrencias.items():
    entidad_a, entidad_b = par
    los_dos_estan_en_el_grafico = entidad_a in posiciones_de_nodos and entidad_b in posiciones_de_nodos
    if los_dos_estan_en_el_grafico:
        x_a, y_a = posiciones_de_nodos[entidad_a]
        x_b, y_b = posiciones_de_nodos[entidad_b]
        eje.plot([x_a, x_b], [y_a, y_b], color="gray", linewidth=veces_juntos * 0.6, alpha=0.5)

# Los nodos, coloreados por categoría y con el tamaño según su frecuencia.
for nodo, coordenadas in posiciones_de_nodos.items():
    x, y = coordenadas
    categoria = categoria_por_entidad[nodo]
    color = colores_por_categoria[categoria]
    tamanio = frecuencia_individual[nodo] * 15
    eje.scatter(x, y, s=tamanio, color=color, zorder=3, edgecolors="white")
    eje.annotate(nodo, (x, y), fontsize=9, ha="center", va="bottom")

for categoria, color in colores_por_categoria.items():
    eje.scatter([], [], color=color, label=categoria.title())

eje.legend(loc="upper right")
eje.set_title("Red de coocurrencia: primeros 1.500 párrafos del corpus")
eje.axis("off")
plt.tight_layout()
plt.show()

### ✎ Para pensar

Mirá qué nodo quedó sin ninguna línea, o con muy pocas — hay al menos uno. ¿Qué te dice eso sobre esa entidad, comparado con una que está en el centro de muchas conexiones? Y al revés: el nodo con más líneas grises alrededor, ¿es también el más grande (el más mencionado), o son cosas distintas?

## Cierre

| Herramienta | Qué te dijo del corpus |
|---|---|
| `pipeline_propio` (`05`) | Qué párrafos mencionan a un personaje, lugar o proyecto de la lista curada |
| `nlp.pipe()` | Cómo procesar 1.500 párrafos sin llamar a `nlp()` uno por uno |
| `Counter` sobre menciones | Quién aparece más en el corpus |
| Pares por párrafo + `Counter` | Quién aparece más seguido junto a quién |
| `matplotlib` + `numpy` | Cómo convertir esa tabla de pares en un mapa de relaciones, sin librería de grafos |

**Limitaciones de este análisis.** La lista de `personajes.txt` está declarada como incompleta — lo que no está en la lista, no entra a la red, aunque aparezca todo el tiempo en el texto. La coocurrencia es por párrafo, no por escena: dos nombres en el mismo párrafo pueden no estar interactuando en absoluto. Y trabajamos sobre los primeros 1.500 párrafos de 5.428 — la parte más temprana de la serie temporal, no el corpus completo.

### Del cuaderno al script

Toda la extracción de coocurrencias también puede vivir fuera de Jupyter, para correrla sobre el corpus completo desde una terminal.

In [ ]:
%%writefile construir_red.py
# ────────────────────────────────────────────────────────────────
# construir_red.py
# Extrae coocurrencias de personajes, lugares y proyectos del corpus
# de Casciari y guarda los quince pares más frecuentes en un CSV.
# Uso: python3 construir_red.py
# ────────────────────────────────────────────────────────────────
import spacy
import pandas as pd
import componentes
from collections import Counter


def extraer_coocurrencias(nlp, textos):
    """Devuelve un Counter con cuántas veces coocurre cada par de entidades."""
    coocurrencias = Counter()
    for doc in nlp.pipe(textos, batch_size=50):
        entidades_del_parrafo = []
        for entidad in doc.ents:
            if entidad.label_ in ("PERSONAJE", "LUGAR", "PROYECTO"):
                entidades_del_parrafo.append(entidad.text)
        entidades_unicas = sorted(set(entidades_del_parrafo))
        for entidad_a in entidades_unicas:
            for entidad_b in entidades_unicas:
                if entidad_a < entidad_b:
                    coocurrencias[(entidad_a, entidad_b)] += 1
    return coocurrencias


if __name__ == "__main__":
    nlp = spacy.load("pipeline_propio")
    corpus = pd.read_json("datos/corpus.jsonl", lines=True)
    textos = list(corpus.head(1500)["texto"])

    coocurrencias = extraer_coocurrencias(nlp, textos)

    filas = []
    for par, veces in coocurrencias.most_common(15):
        entidad_a, entidad_b = par
        filas.append({"entidad_a": entidad_a, "entidad_b": entidad_b, "veces": veces})

    tabla = pd.DataFrame(filas)
    tabla.to_csv("pares_frecuentes.csv", index=False)
    print(tabla)
    print("Guardado en pares_frecuentes.csv")

In [ ]:
# sys.executable: para invocar siempre el mismo Python que ya tiene
# spaCy instalado, esté o no activado el entorno virtual en la terminal.
import sys

!{sys.executable} construir_red.py

### ✎ Para pensar

- `extraer_coocurrencias` recibe `nlp` como parámetro en vez de cargarlo adentro de la función. ¿Qué ganás con eso, pensando en reutilizar esta función sobre otro pipeline?
- Este script ya corre sobre el corpus completo, no solo sobre 1.500 párrafos. ¿Cambiarías algo del análisis si lo corrieras así? (pista: date una vuelta por `06_escala_pipe_y_docbin` antes de responder).